# Task Statement 1.2: Orchestrate Multi-Agent Systems with Coordinator-Subagent Patterns

## 📌 Executive Overview: Goal, Scope & Exam Importance

### Certification Domain & Weight
- **Domain**: Domain 1: Agentic Architecture & Orchestration
- **Domain Weight**: 27% of total CCAR-F exam score
- **Scenario Alignment**: Scenario 3 (Multi-Agent Research System)

### The Core Goal
To design and implement hierarchical multi-agent architectures using the **Hub-and-Spoke** pattern. An architect must enforce strict subagent **context isolation**, implement dynamic query-complexity routing, and construct iterative gap-detection and refinement loops.

### Why This Matters in Production
When a single agent is tasked with handling web research, code analysis, numerical calculations, and executive synthesis in a single conversation context, it quickly suffers from **attention dilution** and catastrophic token bloat. 
Conversely, naive multi-agent systems often connect agents in a peer-to-peer "mesh" where subagents communicate directly with one another. In production, mesh architectures lead to circular invocation loops, uncontrolled token burn, non-deterministic race conditions, and un-auditable outputs. The Hub-and-Spoke pattern provides a deterministic, centrally audited hierarchy where a coordinator directs specialized workers and guarantees quality.

### Exam Importance
Task Statement 1.2 questions frequently test whether candidates understand why subagents must not inherit parent conversation history, why peer-to-peer agent mesh networks are rejected, and how a coordinator detects and remediates synthesis coverage gaps before returning to the user.

---

## 📖 Deep Narrative: Why We Do What We Do

### 1. The Problem: Mesh Networks and Context Bleed
Consider what happens when multi-agent systems are designed without strict topology:
1. **Peer-to-Peer Mesh Anti-Pattern**: If Agent A can directly message Agent B, and Agent B can call Agent C, who can call Agent A, the system is vulnerable to circular recursion. Debugging is impossible because no single entity owns the conversation state or total budget.
2. **Context Bleed (Inheritance Anti-Pattern)**: If a specialized subagent (e.g. searching SEC filings) inherits the coordinator's entire 30,000-token conversational history, its reasoning becomes distracted by irrelevant previous topics, and token costs scale quadratically ($O(N^2)$).

### 2. The Architectural Solution: Hub-and-Spoke with Context Isolation
In production, we enforce three architectural invariants:
1. **Strict Hub-and-Spoke**: All inter-agent communication flows exclusively through the central coordinator. Subagent A never talks to Subagent B. If Subagent B needs Subagent A's findings, the coordinator extracts those findings and explicitly passes them into Subagent B's prompt.
2. **The Context Isolation Contract**: Every subagent is invoked with a **fresh, pristine context**. It receives only:
   - Its specialized system prompt (e.g., "You are an SEC financial analyst").
   - The specific input parameters needed for its task (e.g., `doc_id` and `target_metric`).
   - The minimal toolset required for its domain.
3. **Dynamic Complexity Routing**: The coordinator inspects user query intent:
   - For simple retrieval queries, it answers directly or calls a lightweight tool.
   - For deep multi-faceted research, it dispatches tasks to specialized subagents.
4. **Iterative Refinement & Gap Detection**: Once subagents report their structured findings, the coordinator checks whether all required aspects of the user query have been addressed. If gaps exist (e.g. market data was gathered, but regulatory filings were missing), the coordinator re-delegates targeted follow-up queries before finalizing the output.

---

## 🔑 Key Technical Terminology Explained

- **Hub-and-Spoke Topology**: An architectural pattern where a central Coordinator (the "Hub") manages all subagents (the "Spokes"). The Hub dispatches tasks, collects results, arbitrates errors, and performs final synthesis.
- **Context Isolation Contract**: The rule that subagents operate in sandboxed conversation sessions. They do **not** automatically inherit parent memory, global variables, or unrelated message turns.
- **Dynamic Complexity Routing**: Evaluating the semantic complexity of a user request to decide dynamically which subagents to spawn, avoiding rigid monolithic pipelines.
- **Coverage Gap Detection**: Programmatic or coordinator-evaluated checks to verify that subagent outputs satisfy all criteria in the user request prior to task completion.
- **Attribution Metadata**: Structured data (URL, doc title, section number, publication timestamp) preserved separately from raw text so the coordinator can cite facts accurately.

---

## 🏗️ Architecture: Hub-and-Spoke Topology

```mermaid
flowchart TD
    UserQuery["User Input Query"] --> Coordinator["Coordinator Agent<br>(claude-sonnet-4-5)"]
    
    Coordinator --> RouteDecision{"Query Complexity<br>Router"}
    
    RouteDecision -->|"Simple Retrieval"| SearchSubagent["Web Search Subagent<br>(Isolated Context)"]
    RouteDecision -->|"Code Analysis"| CodeSubagent["Code Analysis Subagent<br>(Isolated Context)"]
    RouteDecision -->|"Data Synthesis"| SynthesisSubagent["Synthesis Subagent<br>(Isolated Context)"]
    
    SearchSubagent -->|"Structured Findings"| Coordinator
    CodeSubagent -->|"Structured AST & Tests"| Coordinator
    SynthesisSubagent -->|"Draft Report"| Coordinator
    
    Coordinator --> GapCheck{"Coverage Gap<br>Detected?"}
    GapCheck -->|"Yes: Re-delegate"| SearchSubagent
    GapCheck -->|"No: Complete"| FinalOutput["Final Synthesized Solution"]
```

---

## ⚠️ Critical Exam Distinctions & Distractor Teardown

| Architectural Pattern | Correct Implementation | Exam Trap to Reject | Why the Trap Fails |
|---|---|---|---|
| **Inter-Agent Communication** | **Hub-and-Spoke**: All inter-agent traffic flows through the coordinator. | **TRAP**: Peer-to-peer mesh topology where subagents invoke each other directly. | Leads to circular recursion, deadlocks, untraceable token spend, and fragmented audit trails. |
| **Context Sharing** | **Context Isolation**: Each subagent starts with a pristine context containing only its task payload. | **TRAP**: Passing the coordinator's full conversational history into every spawned subagent. | Causes attention dilution, context window exhaustion, and quadratic token cost explosions. |
| **Execution Pipeline** | **Dynamic Complexity Routing**: Coordinator routes queries based on semantic requirements. | **TRAP**: A rigid sequential waterfall that passes every query through all subagents. | Wastes latency and token budget on queries that only require one specific subagent. |
| **Output Delivery** | **Iterative Gap Evaluation**: Coordinator verifies synthesis coverage before returning to the user. | **TRAP**: Directly streaming the first subagent's raw draft to the end user. | Delivers incomplete or unverified facts, failing production quality SLAs. |

---

## 💡 Architectural Decision Matrix: Rules of Thumb

| Architectural Dilemma | Recommended Choice | Rapid 30-Second Decision Rule |
|---|---|---|
| **Mesh vs Hub-and-Spoke?** | Hub-and-Spoke | If subagents need to collaborate, route through a coordinator. Never allow peer-to-peer subagent messaging. |
| **Pass full history or task snippet?** | Isolated task snippet | Always isolate subagent context. Pass only the explicit instruction and necessary facts. |
| **How to allocate models?** | Sonnet for Coordinator, Haiku/Sonnet for Workers | Use `claude-sonnet-4-5` for high-level coordination and routing; use `claude-haiku-4-5` for fast, bounded leaf tasks. |

---

## 🧠 Claude 4.5 Extended Thinking & Parameter Standards

In multi-agent coordinator patterns:
- **Coordinator Reasoning**: The coordinator benefits significantly from Extended Thinking (`budget_tokens: 2048` or `4096`) to analyze query intent, assess candidate subagents, and detect coverage gaps.
- **Sampling Constraint**: Omit `temperature`, `top_p`, and `top_k` on both coordinator and subagents when `thinking` is enabled.
- **Thinking Continuity**: In the coordinator's multi-turn loop, assistant thinking blocks from previous routing decisions must be preserved with signatures when dispatching subsequent tools.

---

## 🛠️ Hands-on Workbook Guide: Following the Code Step-by-Step

When exploring [`ts_1_2_coordinator_subagents.ipynb`](ts_1_2_coordinator_subagents.ipynb):
1. **Step 1: Environment & Setup**: Sets up the offline test harness and response audit loggers.
2. **Step 2: Isolated Subagent Classes**: Implements `WebSearchSubagent`, `DocumentAnalysisSubagent`, and `SynthesisSubagent` with isolated scopes.
3. **Step 3: Hub-and-Spoke Coordinator**: Builds `MultiAgentCoordinator` with tool dispatching, centralized audit logging, and gap detection.
4. **Step 4: Multi-Agent Research Simulation**: Runs an end-to-end multi-agent research workflow, verifying coverage status and isolated execution.
5. **Step 5: Verification & Architectural Assertions**: Verifies that subagents never share context memory and all delegation is audited.

In [ ]:
# Step 1: Environment & Offline Setup
import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lib.mock_claude import MockClaudeClient, MockContentBlock, MockMessageResponse

print(" Environment initialized for TS 1.2 Coordinator-Subagent Pattern.")

### 👥 Step 2: Specialized Subagent Definitions (Isolated Contexts)
Each subagent has a dedicated specialization and receives only the explicit parameters required for its job.

In [ ]:
# Step 2: Isolated Subagent Classes
class WebSearchSubagent:
    """Subagent specialized in retrieving external research snippets."""
    KB = {
        "claude agent sdk": "Claude Agent SDK provides primitives for building agentic loops, subagent delegation, hooks, and session resumption.",
        "mcp error handling": "MCP errors should return isError: true with categorized codes (transient, validation, business).",
    }

    def execute(self, query: str) -> Dict[str, Any]:
        q = query.lower()
        matched = []
        for key, val in self.KB.items():
            if any(w in q for w in key.split()):
                matched.append({"topic": key, "snippet": val})
        return {
            "source": "web_search",
            "query": query,
            "found": len(matched) > 0,
            "findings": matched if matched else "No matching external articles.",
        }


class DocumentAnalysisSubagent:
    """Subagent specialized in deep document extraction."""
    def execute(self, doc_id: str, focus_topic: str) -> Dict[str, Any]:
        return {
            "source": "doc_analysis",
            "doc_id": doc_id,
            "focus": focus_topic,
            "metrics": [
                {"metric": "Passing score", "value": "720/1000"},
                {"metric": "Question count", "value": "60 items"},
            ],
        }


class SynthesisSubagent:
    """Subagent specialized in synthesizing multi-source findings."""
    def execute(self, findings: List[Dict[str, Any]], required_topics: List[str]) -> Dict[str, Any]:
        text_dump = json.dumps(findings).lower()
        missing = [t for t in required_topics if t.lower() not in text_dump]
        return {
            "coverage_sufficient": len(missing) == 0,
            "missing_topics": missing,
            "synthesized_findings_count": len(findings),
        }

print(" Subagents declared with isolated execution scopes.")

### 🎛️ Step 3: Multi-Agent Coordinator Implementation
The coordinator orchestrates subagent calls, manages the audit trail, and checks synthesis completeness.

In [ ]:
# Step 3: Hub-and-Spoke Coordinator
class MultiAgentCoordinator:
    def __init__(self, client: Any) -> None:
        self.client = client
        self.web_agent = WebSearchSubagent()
        self.doc_agent = DocumentAnalysisSubagent()
        self.synth_agent = SynthesisSubagent()
        self.audit_log: List[Dict[str, Any]] = []

    def dispatch_tool(self, tool_name: str, inputs: Dict[str, Any]) -> Dict[str, Any]:
        self.audit_log.append({"tool": tool_name, "inputs": inputs})
        if tool_name == "delegate_web_search":
            return self.web_agent.execute(inputs["query"])
        elif tool_name == "delegate_doc_analysis":
            return self.doc_agent.execute(inputs["doc_id"], inputs["focus_topic"])
        elif tool_name == "evaluate_synthesis":
            return self.synth_agent.execute(inputs["findings"], inputs["required_topics"])
        raise ValueError(f"Unknown coordinator tool: {tool_name}")

    def run_research_pipeline(self, topic: str, required_topics: List[str]) -> Dict[str, Any]:
        # Turn 1: Delegate search
        search_res = self.dispatch_tool("delegate_web_search", {"query": topic})
        
        # Turn 2: Delegate doc analysis
        doc_res = self.dispatch_tool("delegate_doc_analysis", {"doc_id": "DOC-2026", "focus_topic": topic})
        
        # Turn 3: Evaluate synthesis coverage
        synth_res = self.dispatch_tool("evaluate_synthesis", {
            "findings": [search_res, doc_res],
            "required_topics": required_topics
        })

        return {
            "status": "completed" if synth_res["coverage_sufficient"] else "gaps_identified",
            "search_res": search_res,
            "doc_res": doc_res,
            "synthesis": synth_res,
            "audit_log": self.audit_log
        }

print(" MultiAgentCoordinator ready for pipeline execution.")

### 🧪 Step 4: Offline Simulation Walkthrough
We execute the coordinator pipeline on a multi-domain research query and observe context isolation and audit logging.

In [ ]:
# Step 4: Run Coordinator Simulation
coord = MultiAgentCoordinator(client=None)

# Run research pipeline
results = coord.run_research_pipeline(
    topic="claude agent sdk",
    required_topics=["claude agent sdk"]
)

print("Pipeline Status:", results["status"])
print("Coverage Sufficient:", results["synthesis"]["coverage_sufficient"])
print("Audit Trail Steps:", len(results["audit_log"]))

### 🔬 Step 5: Verification Assertions
We assert that:
1. Subagents do not cross-contaminate state.
2. Missing topics are accurately flagged during synthesis.
3. Every delegated action is captured in the coordinator's audit log.

In [ ]:
# Step 5: Verification Checks
# 1. Verify complete synthesis coverage
assert results["status"] == "completed"
assert results["synthesis"]["coverage_sufficient"] is True

# 2. Test gap detection with a missing topic
gap_check = coord.synth_agent.execute([results["search_res"]], required_topics=["missing_special_topic"])
assert gap_check["coverage_sufficient"] is False
assert "missing_special_topic" in gap_check["missing_topics"]

# 3. Verify audit log captures all 3 turns
assert len(results["audit_log"]) == 3

print(" All verification assertions passed! Subagent isolation confirmed.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Subagents must **never** share state, conversation history, or memory. They receive explicit task prompts.
- **Rule 2**: All orchestration and routing happens in the coordinator (hub-and-spoke), never peer-to-peer.
- **Rule 3**: The coordinator must evaluate intermediate outputs against goals and re-delegate when gaps exist.